Summarizing Greek available from Perseus and from the First Thousands Years of Greek

In [13]:
import re
repodir = '/Users/gcrane/github/canonical-greekLit'
#repodir = '/Users/gcrane/github/canonical-latinLit'
#fname = '/Users/gcrane/github/First1KGreek'

curlang = 'grc'
curlang = 'eng'


# a very simple word counter:
# It strips out tags and at least some notes.
# Then it counts space delimeted string that are left.
# It does wait until it sees a <text/> tag before starting to count.

def countwords(fname):
    #print('counting',fname)
    f = open(fname)
    intext = 0
    totwords = 0
    for l in f:
        if(re.search('<text',l)):
            #print('intxt',l)
            intext = 1
            continue
        if(not intext):
            continue
        #print(l)
        l = re.sub('<note[^>]*>.+</note>',' ',l)
        l = re.sub('<[^>]+>',' ',l)
        totwords = totwords + len((l.split()))
    return(totwords)




In [14]:
import os
import re
worksizes = {}


## first load up a list of Greek texts that are parsing --
## This needs to be updated

f = open('cangkbuild-2022-11-21.txt')
passedfiles = []
passedworks = []
for l in f:
    l = re.sub('\n','',l)
    m = re.search('/(tlg[^/]+)(\.[^\.]+\.xml)$',l)
    if( not m):
        continue
    if(m):
        passedfiles.append(m[1]+m[2])
        if(not m[1] in passedworks):
            passedworks.append(m[1])

## this directory will obviously differ            
#fname = '/Users/gcrane/github/canonical-greekLit'
#fname = '/Users/gcrane/github/First1KGreek'
failedfiles = {}
totfails = 0
totengfails = 0
toteng = 0
totgreek = 0

failedauths_e = {}
failedauths_g = {}

greekauthtots = {}
        
    
f.close()

tg_namelist = {}
tg_namelist['tlg3135'] = 'Johannes Zonaras'
## extract author and title from TEI XML
def get_tg_name(fname):
    f = open(fname)
    #print(fname)
    curtgid = ''
    curname = ''
    
    for l in f:
        m = re.search('(tlg[0-9]+)',l)
        if(m):
            curtgid = m[1]
        m = re.search('<(ti:groupname[^>]*|cts:groupname[^>]*)>([^<]+)',l)
        if(m):
            curname = m[2]
            #print('hit',l)
        #else:
        #    print('fail',l)
    tg_namelist[curtgid] = curname
    return(curname)


for top, dirs, files in os.walk(repodir):
    for file in files:
        if(file == 'tlg0099.tlg001.perseus-grc1.xml'):
            print('skipping',file)
            continue
        
        ## where we have metadata files for a text group, 
        ## set the author name for each text group ID.
        
        
        if(re.search('data/tlg[0-9]+/__cts__.xml',os.path.join(top, file))):
            tg_name = get_tg_name(os.path.join(top, file))
            continue
            
        ## now only examine XML files marked as being Greek
        if( not re.search('^tlg[^/]+-'+curlang+'[0-9]\.xml$',file)):
            continue
            
        ## get the text group and work ids
        m = re.search('(tlg[0-9][0-9][0-9][0-9])\.(tlg[0-9][0-9][0-9])',file)
        curtlg = ''
        curworkid = ''
        if(m):
            curtlg = m[1]
            curworkid = m[2]
            curkey = curtlg + '.' + curworkid
        else:
            print('fail',file)

        ## count the words in this work
        cursize = countwords(os.path.join(top, file))
        ## sometimes we have multiple editions, so don't count any 
        ## extra editions of the same work
        if(not curkey in worksizes):
            worksizes[curkey] = cursize
            if(re.search('tlg0099',curkey)):
                print('tlg0099',curkey,worksizes[curkey])
        #if(not file in passedfiles and not re.sub('1\.xml','2.xml',file) in passedfiles):
        if(not file in passedfiles and not re.sub('1\.xml','2.xml',file) in passedfiles):
            totfails = totfails + cursize
            failedfiles[file] = cursize
            #print(os.path.getsize(os.path.join(top, file)),os.path.join(top, file))
        else:
            totgreek = totgreek + cursize
        if(not re.search('-'+curlang,file)):
            continue
        if(curtlg in greekauthtots):
            greekauthtots[curtlg] = greekauthtots[curtlg] + cursize
        else:
            greekauthtots[curtlg] = cursize
            
print(totfails,len(failedfiles))
print(curlang + ' hits and fails',totgreek,totfails)

i = 0
sofar = 0

print(curlang)
for foo in sorted(failedfiles,key=failedfiles.get,reverse=True):
    i = i + 1
    sofar = sofar + failedfiles[foo]
    perc = re.sub('(\.[0-9][0-9]).+','\g<1>',str(sofar/totfails))
    print(i, failedfiles[foo],sofar,perc,foo)
    
    


0 0
eng hits and fails 0 0
eng


In [15]:
print(len(greekauthtots))

for foo in sorted(greekauthtots,key=greekauthtots.get,reverse=True):
    if(not foo in tg_namelist):
        tg_namelist[foo] = 'na'
    curname = tg_namelist[foo]
      
    print(foo,greekauthtots[foo],curname)

      

0


In [16]:
genretable = {
'tlg3135' : 'prose', # Joannes Zonaras
'tlg0059' : 'prose', # Plato
'tlg0032' : 'prose', # Xenophon
'tlg0035' : 'poetry', # Moschus
'tlg0003' : 'prose', # Thucydides
'tlg0004' : 'prose', # Diogenes Laertius
'tlg0058' : 'prose', # Aeneas Tacticus
'tlg0093' : 'prose', # Theophrastus
'tlg0060' : 'prose', # Diodorus Siculus
'tlg1389' : 'prose', # Harpocration, Valerius
'tlg0638' : 'prose', # Philostratus the Athenian
'tlg1311' : 'prose', # Didache
'tlg0033' : 'poetry', # Pindar
'tlg0525' : 'prose', # Pausanias
'tlg0719' : 'prose', # Aretaeus of Cappadocia
'tlg1600' : 'prose', # Philostratus the Lemnian (Philostratus Major)
'tlg0540' : 'prose', # Lysias
'tlg0548' : 'prose', # Apollodorus
'tlg1271' : 'prose', # Clemens Romanus (Clement of Rome)
'tlg0554' : 'prose', # Chariton
'tlg0562' : 'prose', # Marcus Aurelius
'tlg1443' : 'prose', # Ignatius of Antioch
'tlg1622' : 'prose', # Polycarp
'tlg1419' : 'prose', # Hermas, 2nd cent.
'tlg0011' : 'poetry', # Sophocles
'tlg0016' : 'prose', # Herodotus
'tlg0020' : 'poetry', # Hesiod
'tlg0074' : 'prose', # Arrian
'tlg0019' : 'poetry', # Aristophanes
'tlg0028' : 'prose', # Antiphon
'tlg0641' : 'prose', # Xenophon of Ephesus
'tlg0010' : 'prose', # Isocrates
'tlg0646' : 'prose', # Pseudo-Justinus Martyr
'tlg0612' : 'prose', # Dio Chrysostom
'tlg0086' : 'prose', # Aristotle
'tlg0031' : 'prose', # New Testament
'tlg0007' : 'prose', # Plutarch
'tlg0062' : 'prose', # Lucian
'tlg0001' : 'poetry', # Apollonius Rhodius
'tlg0006' : 'poetry', # Euripides
'tlg4036' : 'prose', # Proclus
'tlg0008' : 'prose', # Athenaeus
'tlg0090' : 'prose', # Agathemerus
'tlg0099' : 'prose', # Strabo
'tlg0543' : 'prose', # Polybius
'tlg0527' : 'prose', # Old Testament
'tlg0560' : 'prose', # Longinus
'tlg0551' : 'prose', # Appian
'tlg1484' : 'prose', # Martyrium Polycarpi
'tlg0557' : 'prose', # Epictetus
'tlg1216' : 'prose', # Barnabas
'tlg0561' : 'prose', # Longus
'tlg0012' : 'poetry', # Homer
'tlg0023' : 'poetry', # Oppian
'tlg0085' : 'poetry', # Aeschylus
'tlg1799' : 'prose', # Euclid
'tlg0013' : 'poetry', # Homeric Hymns
'tlg0014' : 'prose', # Demosthenes
'tlg0526' : 'prose', # Flavius Josephus
'tlg0081' : 'prose', # Dionysius of Halicarnassus
'tlg0284' : 'prose', # Aristides, Aelius
'tlg4029' : 'prose', # Procopius
'tlg0385' : 'prose', # Cassius Dio Cocceianus
'tlg0545' : 'prose', # Aelian
'tlg7000' : 'poetry', # Greek Anthology
'tlg2045' : 'poetry', # Nonnus of Panopolis
'tlg2003' : 'prose', # Julian the Emperor
'tlg0627' : 'prose', # Hippocrates
'tlg2018' : 'prose', # Eusebius of Caesarea
'tlg2934' : 'prose', # John, of Damascus (attributed author)
'tlg2040' : 'prose', # Basil, Saint, Bishop of Caesarea
'tlg2046' : 'poetry', # Quintus Smyrnaeus
'tlg0026' : 'prose', # Aeschines
'tlg0532' : 'prose', # Achilles Tatius
'tlg0363' : 'prose', # Claudius Ptolemy
'tlg0017' : 'prose', # Isaeus
'tlg0555' : 'prose', # Clement of Alexandria
'tlg0057' : 'prose', # Galen
'tlg0094' : 'prose', # Pseudo-Plutarch
'tlg0005' : 'poetry', # Theocritus
'tlg0027' : 'prose', # Andocides
'tlg0533' : 'poetry', # Callimachus
'tlg0030' : 'prose', # Hyperides
'tlg0613' : 'prose', # Demetrius of Phaleron (attributed author)
'tlg0024' : 'poetry', # Oppian of Apamea
'tlg0029' : 'prose', # Dinarchus
'tlg0034' : 'prose', # Lycurgus
'tlg0648' : 'prose', # Onasander
'tlg0653' : 'poetry', # Aratus Solensis
'tlg0341' : 'poetry', # Lycophron
'tlg0652' : 'prose', # Philostratus Minor
'tlg0556' : 'prose', # Asclepiodotus
'tlg0655' : 'prose', # Parthenius
'tlg0199' : 'poetry', # Bacchylides
'tlg0647' : 'poetry', # Tryphiodorus
'tlg4091' : 'prose', # Callistratus
'tlg4081' : 'poetry', # Colluthus
'tlg0036' : 'poetry', # Bion of Phlossa
'tlg0535' : 'prose' # Demades
}

tg_namelist = {
    'tlg3135' : 'Joannes Zonaras',
'tlg0059' : 'Plato',
'tlg0032' : 'Xenophon',
'tlg0035' : 'Moschus',
'tlg0003' : 'Thucydides',
'tlg0004' : 'Diogenes Laertius',
'tlg0058' : 'Aeneas Tacticus',
'tlg0093' : 'Theophrastus',
'tlg0060' : 'Diodorus Siculus',
'tlg1389' : 'Harpocration, Valerius',
'tlg0638' : 'Philostratus the Athenian',
'tlg1311' : 'Didache',
'tlg0033' : 'Pindar',
'tlg0525' : 'Pausanias',
'tlg0719' : 'Aretaeus of Cappadocia',
'tlg1600' : 'Philostratus the Lemnian (Philostratus Major)',
'tlg0540' : 'Lysias',
'tlg0548' : 'Apollodorus',
'tlg1271' : 'Clemens Romanus (Clement of Rome)',
'tlg0554' : 'Chariton',
'tlg0562' : 'Marcus Aurelius',
'tlg1443' : 'Ignatius of Antioch',
'tlg1622' : 'Polycarp',
'tlg1419' : 'Hermas, 2nd cent.',
'tlg0011' : 'Sophocles',
'tlg0016' : 'Herodotus',
'tlg0020' : 'Hesiod',
'tlg0074' : 'Arrian',
'tlg0019' : 'Aristophanes',
'tlg0028' : 'Antiphon',
'tlg0641' : 'Xenophon of Ephesus',
'tlg0010' : 'Isocrates',
'tlg0646' : 'Pseudo-Justinus Martyr',
'tlg0612' : 'Dio Chrysostom',
'tlg0086' : 'Aristotle',
'tlg0031' : 'New Testament',
'tlg0007' : 'Plutarch',
'tlg0062' : 'Lucian',
'tlg0001' : 'Apollonius Rhodius',
'tlg0006' : 'Euripides',
'tlg4036' : 'Proclus',
'tlg0008' : 'Athenaeus',
'tlg0090' : 'Agathemerus',
'tlg0099' : 'Strabo',
'tlg0543' : 'Polybius',
'tlg0527' : 'Old Testament',
'tlg0560' : 'Longinus',
'tlg0551' : 'Appian',
'tlg1484' : 'Martyrium Polycarpi',
'tlg0557' : 'Epictetus',
'tlg1216' : 'Barnabas',
'tlg0561' : 'Longus',
'tlg0012' : 'Homer',
'tlg0023' : 'Oppian',
'tlg0085' : 'Aeschylus',
'tlg1799' : 'Euclid',
'tlg0013' : 'Homeric Hymns',
'tlg0014' : 'Demosthenes',
'tlg0526' : 'Flavius Josephus',
'tlg0081' : 'Dionysius of Halicarnassus',
'tlg0284' : 'Aristides, Aelius',
'tlg4029' : 'Procopius',
'tlg0385' : 'Cassius Dio Cocceianus',
'tlg0545' : 'Aelian',
'tlg7000' : 'Greek Anthology',
'tlg2045' : 'Nonnus of Panopolis',
'tlg2003' : 'Julian the Emperor',
'tlg0627' : 'Hippocrates',
'tlg2018' : 'Eusebius of Caesarea',
'tlg2934' : 'John, of Damascus (attributed author)',
'tlg2040' : 'Basil, Saint, Bishop of Caesarea',
'tlg2046' : 'Quintus Smyrnaeus',
'tlg0026' : 'Aeschines',
'tlg0532' : 'Achilles Tatius',
'tlg0363' : 'Claudius Ptolemy',
'tlg0017' : 'Isaeus',
'tlg0555' : 'Clement of Alexandria',
'tlg0057' : 'Galen',
'tlg0094' : 'Pseudo-Plutarch',
'tlg0005' : 'Theocritus',
'tlg0027' : 'Andocides',
'tlg0533' : 'Callimachus',
'tlg0030' : 'Hyperides',
'tlg0613' : 'Demetrius of Phaleron (attributed author)',
'tlg0024' : 'Oppian of Apamea',
'tlg0029' : 'Dinarchus',
'tlg0034' : 'Lycurgus',
'tlg0648' : 'Onasander',
'tlg0653' : 'Aratus Solensis',
'tlg0341' : 'Lycophron',
'tlg0652' : 'Philostratus Minor',
'tlg0556' : 'Asclepiodotus',
'tlg0655' : 'Parthenius',
'tlg0199' : 'Bacchylides',
'tlg0647' : 'Tryphiodorus',
'tlg4091' : 'Callistratus',
'tlg4081' : 'Colluthus',
'tlg0036' : 'Bion of Phlossa',
'tlg0535' : 'Demades'
}

In [17]:

def getauthwork(fname):
    retauth = ''
    retwork = ''
    f = open(fname)
    for l in f:
        if(retauth and retwork):
            break
        m = re.search('<author[^>]*>([^<]+)',l)
        if(not retauth and m):
            retauth = m[1]
            #print('author',retauth)
        m = re.search('<title\\b[^>]*>([^<]+)',l)
        if(not retwork and m):
            retwork = m[1]
            if(re.search('New Testament',retwork)):
                retauth,retwork = retwork.split(' - ')
                break
            if(re.search('IOANNIS ZONARAE EPITOME HISTORIARUM',retwork)):
                retauth = 'Joannes Zonaras'
                retwork = 'Epitome Historiarum'
                print(retauth,retwork)
                break
            #print('work',retwork,l)
    f.close()
    return(retauth,retwork)

i = 0
worklist = []
for top, dirs, files in os.walk(repodir):
    for file in files:
        if(file == 'tlg0099.tlg001.perseus-grc1.xml'):
            print('skipping',file)
            continue
        m = re.search('(tlg[0-9][0-9][0-9][0-9])\.(tlg[0-9][0-9][0-9]).*-grc[0-9]\.xml$',file)
        if(m):
            curauthid = m[1]
            curworkid = m[2]
            if(curauthid == 'tlg0090'):
                curauth = 'Agathemerus'
                curwork = 'Geographiae Informatio'
            else:
                curauth,curwork = getauthwork(os.path.join(top, file))
            if(not curauthid in tg_namelist or re.search('Zonaras',curauth)):
                tg_namelist[curauthid] = curauth
            if(tg_namelist[curauthid] == 'na' or 1):
                i = i + 1
                tg_namelist[curauthid] = curauth
                curkey = curauthid + '.' + curworkid
                if(curkey in passedworks):
                    stat = 'p'
                else:
                    stat = 'f'
                if( curauthid in genretable):
                    prose_poetry = genretable[curauthid]
                else:
                    prose_poetry = 'prose'
                if(curkey in worksizes):
                    cursize = worksizes[curkey]
                else:
                    cursize = 0
                worklist.append([curauthid,curauth,curworkid,curwork,cursize,prose_poetry,stat])
                #print(i,curauthid,curauth,curworkid,curwork,worksizes[curkey],stat)
#worklist
print('tgs',len(tg_namelist))

tgs 99


In [18]:
worksizes

{}

In [19]:
import pandas as pd
labels = ['curauthid','curauth','curworkid','curwork','words','prose/poetry','stat']

df = pd.DataFrame(worklist,columns=labels)
df

,curauthid,curauth,curworkid,curwork,words,prose/poetry,stat


In [20]:
select_pass = df.loc[df['stat'] == 'p']
select_pass

,curauthid,curauth,curworkid,curwork,words,prose/poetry,stat


In [21]:
import plotly.express as px


#fig = px.sunburst(df, path=['curauth','curwork', 'stat'], values='words')
#fig.show()
#fig.write_html('persgreek.html')

#fig = px.treemap(select_pass, path=['prose/poetry','curauth','curwork'], values='words')
fig = px.treemap(df, path=['prose/poetry','curauth','curwork'], values='words')
fig.show()
#fig.write_html('persgreektree.html')

In [9]:
outf = open('genre.tsv','w')
for foo in tg_namelist:
    print(foo,tg_namelist[foo],sep='\t',file=outf)
outf.close()

In [10]:
tg_namelist

{'tlg3135': 'Joannes Zonaras',
 'tlg0059': 'Plato',
 'tlg0032': 'Xenophon',
 'tlg0035': 'Moschus',
 'tlg0003': 'Thucydides',
 'tlg0004': 'Diogenes Laertius',
 'tlg0058': 'Aeneas Tacticus',
 'tlg0093': 'Theophrastus',
 'tlg0060': 'Diodorus Siculus',
 'tlg1389': 'Harpocration, Valerius',
 'tlg0638': 'Philostratus the Athenian',
 'tlg1311': 'Didache',
 'tlg0033': 'Pindar',
 'tlg0525': 'Pausanias',
 'tlg0719': 'Aretaeus of Cappadocia',
 'tlg1600': 'Philostratus the Lemnian (Philostratus Major)',
 'tlg0540': 'Lysias',
 'tlg0548': 'Apollodorus',
 'tlg1271': 'Clemens Romanus (Clement of Rome)',
 'tlg0554': 'Chariton',
 'tlg0562': 'Marcus Aurelius',
 'tlg1443': 'Ignatius of Antioch',
 'tlg1622': 'Polycarp',
 'tlg1419': 'Hermas, 2nd cent.',
 'tlg0011': 'Sophocles',
 'tlg0016': 'Herodotus',
 'tlg0020': 'Hesiod',
 'tlg0074': 'Arrian',
 'tlg0019': 'Aristophanes',
 'tlg0028': 'Antiphon',
 'tlg0641': 'Xenophon of Ephesus',
 'tlg0010': 'Isocrates',
 'tlg0646': 'Pseudo-Justinus Martyr',
 'tlg0612': 'D